# 관계 네트워크 피처 생성 v2 — 가이드 전항목 이행

개발팀(PL) 작업 가이드 · 그룹 3분할본 v2

## v1 대비 추가 (가이드 미이행분 보완)

| 구분 | 추가 내용 | 가이드 근거 |
|---|---|---|
| 기관(근사) 축 | `PRINST_TY` + `PRINST_LOCPLC_SIGNGU_CODE` 조합을 근사 키로 | 2장 주의사항 · 3장 "기관(근사) 중심 피처가 기여 가능성 큼" |
| 지역 축 | `PRINST_LOCPLC_SIGNGU_CODE` 단독 | 3장 "지역 중심 피처" |
| 동시결제 | `DELNG_DT` 기준 시간차 10분 내 타 이용자 결제 | 2장 표 — `DELNG_DT` 용도 |
| C군 | 이분 그래프 사영 후 고유벡터 중심성 · Louvain 커뮤니티 | 4장 C |

**주의**: 기관 근사 키는 정확한 기관 식별이 아니다. 유형×시군구 조합이므로 동일 지역·유형의 복수 기관이 하나로 묶인다.

In [ ]:
#@title ★ 재개 — S1~S6 건너뛰고 바로 실험 상태로
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, sys, json, pickle, warnings, numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_fscore_support
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/부정수급FDS/모델제작-관계 피처 담당자/fds_split_v2'
OUT  = '/content/drive/MyDrive/부정수급FDS/모델제작-관계 피처 담당자/graph_out_v2'

tr = pd.read_parquet(f'{BASE}/train.parquet')
va = pd.read_parquet(f'{BASE}/valid.parquet')
te = pd.read_parquet(f'{BASE}/test.parquet')
for d in (tr, va, te):
    d['_y'] = (d['INJSTUSE_AT'].astype(str).str.upper()=='Y').astype(np.int8) \
          if d['INJSTUSE_AT'].dtype == object else d['INJSTUSE_AT'].astype(np.int8)
    d['_INST']   = d['PRINST_TY'].astype(str)+'|'+d['PRINST_LOCPLC_SIGNGU_CODE'].astype(str)
    d['_REGION'] = d['PRINST_LOCPLC_SIGNGU_CODE'].astype(str)

F = {n: pd.read_parquet(f'{OUT}/graph_feats_{n}.parquet') for n in ['train','valid','test']}
with open(f'{OUT}/graph_feats.pkl','rb') as f: ART = pickle.load(f)
FEAT_ORDER  = ART['feature_order']
RECOMMENDED = ART['recommended']

DROP = {'row_id','_y','_INST','_REGION','INJSTUSE_AT','TRGTER_ID','PRHNF_ID','DELNG_DT',
        'DELNG_DE','PREDELNG_DE','SVC_BEGIN_DT','SVC_END_DT','CONFM_NO','DELNG_SN','PRINST_ID'}
num = [c for c in tr.columns if c not in DROP and pd.api.types.is_numeric_dtype(tr[c])]

_trp = set(map(tuple, tr[['TRGTER_ID','PRHNF_ID']].values))
pair_seen = pd.Series(list(map(tuple, va[['TRGTER_ID','PRHNF_ID']].values))).isin(_trp).values

N_TREES, SEEDS = 600, [42, 7, 2024]
def fit_eval(cols, seed):
    Xtr = pd.concat([tr[num].reset_index(drop=True), F['train'][cols]], axis=1) if cols else tr[num]
    Xva = pd.concat([va[num].reset_index(drop=True), F['valid'][cols]], axis=1) if cols else va[num]
    m = lgb.LGBMClassifier(n_estimators=N_TREES, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, verbose=-1, random_state=seed)
    m.fit(Xtr, tr._y)
    p = m.predict_proba(Xva)[:,1]
    return (average_precision_score(va._y, p),
            average_precision_score(va._y.values[~pair_seen], p[~pair_seen]))

sys.path.insert(0, OUT)
from graph_predict import add_graph_feats

if os.path.exists(f'{OUT}/experiments.pkl'):
    _e = pickle.load(open(f'{OUT}/experiments.pkl','rb'))
    STAT, ST2, SETS, FINAL = _e['STAT'], _e['ST2'], _e['SETS'], _e['FINAL']
    print(f"실험 결과 복원 — STAT {len(STAT)}구성 · ST2 {len(ST2)}구성")

print(f'복귀 완료 · 피처 {len(FEAT_ORDER)}개 · 권고 {RECOMMENDED}')
print(f"조합 미존재 n={(~pair_seen).sum():,} · 양성률 {va._y.values[~pair_seen].mean():.4f}")

---
# S0. 로드 · 진단

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, json, pickle, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/부정수급FDS/모델제작-관계 피처 담당자/fds_split_v2'
OUT  = '/content/drive/MyDrive/부정수급FDS/모델제작-관계 피처 담당자/graph_out_v2'
os.makedirs(OUT, exist_ok=True)

tr = pd.read_parquet(f'{BASE}/train.parquet')
va = pd.read_parquet(f'{BASE}/valid.parquet')
te = pd.read_parquet(f'{BASE}/test.parquet')
MAN = json.load(open(f'{BASE}/split_manifest.json'))

def to_y(s):
    return (s.astype(str).str.upper()=='Y').astype(np.int8).values if s.dtype==object \
           else s.values.astype(np.int8)
for d in (tr, va, te):
    d['_y'] = to_y(d['INJSTUSE_AT'])

print(f'train {tr.shape}  valid {va.shape}  test {te.shape}')
print(f'부정률  {tr._y.mean():.4f} / {va._y.mean():.4f} / {te._y.mean():.4f}')

# 기관 근사 키 재료 확인
for c in ['PRINST_TY','PRINST_LOCPLC_SIGNGU_CODE','DELNG_DT','SVC_BEGIN_DT','SVC_END_DT']:
    print(f'{c:28s} 존재 {c in tr.columns}  고유값 {tr[c].nunique() if c in tr.columns else "-"}')

In [ ]:
#@title S0-2. 축별 노드 중복률 — 어느 축이 분할을 가로지르는가
def mk_keys(d):
    d = d.copy()
    d['_INST'] = (d['PRINST_TY'].astype(str) + '|' + d['PRINST_LOCPLC_SIGNGU_CODE'].astype(str))
    d['_REGION'] = d['PRINST_LOCPLC_SIGNGU_CODE'].astype(str)
    return d
tr, va, te = mk_keys(tr), mk_keys(va), mk_keys(te)

print(f"{'축':16s}{'train 고유':>12}{'valid 존재':>12}{'test 존재':>12}")
for c, nm in [('TRGTER_ID','이용자'), ('PRHNF_ID','제공인력'),
              ('_INST','기관(근사)'), ('_REGION','지역')]:
    s = set(tr[c])
    print(f'{nm:16s}{tr[c].nunique():>12,}{va[c].isin(s).mean():>11.1%}{te[c].isin(s).mean():>12.1%}')

trp = set(map(tuple, tr[['TRGTER_ID','PRHNF_ID']].values))
for nm, d in [('valid',va), ('test',te)]:
    ov = pd.Series(list(map(tuple, d[['TRGTER_ID','PRHNF_ID']].values))).isin(trp).mean()
    print(f'{"조합":16s}{len(trp):>12,}{ov:>11.1%}' if nm=='valid' else f'{"":16s}{"":>12}{"":>12}{ov:>12.1%}')

print('\n→ 기관·지역 축은 노드 수가 적어 분할을 가로질러 거의 100% 존재할 것.')
print('  조합 축(54%)과 성격이 달라 누수 구조가 다름.')

---
# S1. 그래프 구성 — train 행만 (가이드 3장-1)

In [ ]:
E = (tr.groupby(['TRGTER_ID','PRHNF_ID'])
       .agg(pair_cnt=('row_id','size'), pair_pos=('_y','sum')).reset_index())
T = (E.groupby('TRGTER_ID')
       .agg(t_deg=('PRHNF_ID','nunique'), t_cnt=('pair_cnt','sum'), t_pos=('pair_pos','sum')).reset_index())
P = (E.groupby('PRHNF_ID')
       .agg(p_deg=('TRGTER_ID','nunique'), p_cnt=('pair_cnt','sum'), p_pos=('pair_pos','sum')).reset_index())

# 기관(근사) · 지역 노드 테이블
I = (tr.groupby('_INST')
       .agg(i_cnt=('row_id','size'), i_pos=('_y','sum'),
            i_prhnf=('PRHNF_ID','nunique'), i_trgter=('TRGTER_ID','nunique')).reset_index())
R = (tr.groupby('_REGION')
       .agg(r_cnt=('row_id','size'), r_pos=('_y','sum'),
            r_prhnf=('PRHNF_ID','nunique'), r_trgter=('TRGTER_ID','nunique')).reset_index())

print(f'엣지 {len(E):,}  이용자 {len(T):,}  인력 {len(P):,}  기관(근사) {len(I):,}  지역 {len(R):,}')
print(f'기관 결제수 중앙값 {I.i_cnt.median():.0f}  지역 결제수 중앙값 {R.r_cnt.median():.0f}')

---
# S2. A군 — 구조 피처 (라벨 불사용)

In [ ]:
#@title S2-1. HHI · 연결요소
E = E.merge(P[['PRHNF_ID','p_cnt']], on='PRHNF_ID', how='left')
E['sh_p'] = E.pair_cnt / E.p_cnt
P = P.merge(E.groupby('PRHNF_ID')['sh_p'].apply(lambda s: float((s**2).sum()))
             .rename('p_hhi').reset_index(), on='PRHNF_ID', how='left')

from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import connected_components
t_idx = {v:i for i,v in enumerate(T.TRGTER_ID)}
p_idx = {v:i+len(T) for i,v in enumerate(P.PRHNF_ID)}
N = len(T)+len(P)
ri = E.TRGTER_ID.map(t_idx).values; ci = E.PRHNF_ID.map(p_idx).values
A = coo_matrix((np.ones(len(E)), (ri,ci)), shape=(N,N))
ncomp, lab = connected_components(A, directed=False)
sz = np.bincount(lab)
T['comp_id'], P['comp_id'] = lab[:len(T)], lab[len(T):]
T['comp_size'], P['comp_size'] = sz[T.comp_id.values], sz[P.comp_id.values]
print(f'연결요소 {ncomp:,}  최대 {sz.max():,}  중앙값 {np.median(sz):.0f}  크기≤2 {(sz<=2).mean():.1%}')

In [ ]:
#@title S2-2. 서비스 시간 겹침 + 동시결제 (가이드 2장 DELNG_DT 용도)
import heapq
from collections import Counter

# ① 서비스 구간 겹침
x = tr[['PRHNF_ID','TRGTER_ID','SVC_BEGIN_DT','SVC_END_DT']].copy()
x['b'] = pd.to_numeric(x.SVC_BEGIN_DT, errors='coerce')
x['e'] = pd.to_numeric(x.SVC_END_DT,   errors='coerce')
x = x.dropna(subset=['b','e']); x = x[x.e >= x.b]
ov = {}
for p, g in x.groupby('PRHNF_ID', sort=False):
    if len(g) < 2: ov[p]=0; continue
    g = g.sort_values('b'); bb,ee,tt = g.b.values, g.e.values, g.TRGTER_ID.values
    heap, per, alive, tot = [], Counter(), 0, 0
    for i in range(len(g)):
        while heap and heap[0][0] < bb[i]:
            _, t0 = heapq.heappop(heap); per[t0]-=1; alive-=1
        tot += alive - per[tt[i]]
        heapq.heappush(heap, (ee[i], tt[i])); per[tt[i]]+=1; alive+=1
    ov[p] = int(tot)
P['p_time_ov'] = P.PRHNF_ID.map(ov).fillna(0).astype(np.int32)

# ② DELNG_DT 기준 동시결제 (10분 내 서로 다른 이용자)
WIN = 600  # 초
z = tr[['PRHNF_ID','TRGTER_ID','DELNG_DT']].copy()
z['ts'] = pd.to_datetime(z.DELNG_DT, errors='coerce', utc=True).astype('int64') // 10**9
z = z.dropna(subset=['ts'])
sp = {}
for p, g in z.groupby('PRHNF_ID', sort=False):
    if len(g) < 2: sp[p]=0; continue
    g = g.sort_values('ts'); ts, tt = g.ts.values, g.TRGTER_ID.values
    lo, c, per, alive = 0, 0, Counter(), 0
    for i in range(len(g)):
        while ts[i] - ts[lo] > WIN:
            per[tt[lo]] -= 1; alive -= 1; lo += 1
        c += alive - per[tt[i]]
        per[tt[i]] += 1; alive += 1
    sp[p] = int(c)
P['p_same_pay'] = P.PRHNF_ID.map(sp).fillna(0).astype(np.int32)

print(f'서비스 겹침 발생 인력 {(P.p_time_ov>0).sum():,}명 ({(P.p_time_ov>0).mean():.1%})  최대 {P.p_time_ov.max():,}')
print(f'동시결제 발생 인력   {(P.p_same_pay>0).sum():,}명 ({(P.p_same_pay>0).mean():.1%})  최대 {P.p_same_pay.max():,}')

In [ ]:
#@title S2-3. C군 — 사영 그래프 중심성 + Louvain (가이드 4장 C)
from scipy.sparse.linalg import eigsh

B = csr_matrix((np.ones(len(E)), (E.TRGTER_ID.map(t_idx).values,
                                  E.PRHNF_ID.map(p_idx).values - len(T))),
               shape=(len(T), len(P)))

def eig_cent(M):
    """대칭 사영행렬의 최대 고유벡터 (절댓값 정규화)"""
    M = M.astype(np.float64)
    M.setdiag(0); M.eliminate_zeros()
    if M.shape[0] < 3 or M.nnz == 0:
        return np.zeros(M.shape[0], dtype=np.float32)
    try:
        v = eigsh(M, k=1, which='LA', maxiter=3000, tol=1e-4)[1][:,0]
        v = np.abs(v); m = v.max()
        return (v/m).astype(np.float32) if m > 0 else v.astype(np.float32)
    except Exception as ex:
        print('  eigsh 실패 → 차수로 대체:', ex)
        d = np.asarray(M.sum(1)).ravel(); m = d.max()
        return (d/m).astype(np.float32) if m > 0 else d.astype(np.float32)

Pp = (B.T @ B).tocsr()   # 인력–인력 (같은 이용자 공유)
Tp = (B @ B.T).tocsr()   # 이용자–이용자 (같은 인력 공유)
print(f'사영 nnz — 인력 {Pp.nnz:,}  이용자 {Tp.nnz:,}')
P['p_eig'] = eig_cent(Pp)
T['t_eig'] = eig_cent(Tp)

# Louvain — 연결요소가 잘게 쪼개져 있어 크기 10 이상만 적용
import networkx as nx
big = set(np.where(sz >= 10)[0])
com_id = lab.copy().astype(np.int64)          # 기본: 연결요소 = 커뮤니티
if big:
    sub = E[E.TRGTER_ID.map(t_idx).map(lambda i: lab[i] in big)]
    G = nx.Graph(); G.add_edges_from(zip(sub.TRGTER_ID.map(t_idx), sub.PRHNF_ID.map(p_idx)))
    if G.number_of_nodes():
        parts = nx.community.louvain_communities(G, seed=42)
        off = com_id.max() + 1
        for k, cset in enumerate(parts):
            for n_ in cset: com_id[n_] = off + k
        print(f'Louvain 적용 — 대형 연결요소 {len(big)}개 → 커뮤니티 {len(parts):,}개')
csz = pd.Series(com_id).value_counts()
T['com_id'], P['com_id'] = com_id[:len(T)], com_id[len(T):]
T['com_size'] = T.com_id.map(csz).values; P['com_size'] = P.com_id.map(csz).values
print(f'최종 커뮤니티 {csz.size:,}개  최대 {csz.max():,}  중앙값 {csz.median():.0f}')

---
# S3. B군 — 라벨 사용 (가이드 3장-4 leave-one-out)

노드별 `cnt`/`pos`를 저장하고, 적용 시점에 자기 조합 몫을 빼는 방식.

In [ ]:
GLOBAL_POS = float(tr._y.mean())

# 연결요소 · 커뮤니티 단위 집계
EE = E.merge(T[['TRGTER_ID','comp_id','com_id']], on='TRGTER_ID', how='left')
cp  = EE.groupby('comp_id').agg(c_cnt=('pair_cnt','sum'), c_pos=('pair_pos','sum')).reset_index()
cm  = EE.groupby('com_id').agg(m_cnt=('pair_cnt','sum'), m_pos=('pair_pos','sum')).reset_index()
T = T.merge(cp, on='comp_id', how='left').merge(cm, on='com_id', how='left')
P = P.merge(cp, on='comp_id', how='left').merge(cm, on='com_id', how='left')

print('노드 테이블 완성')
for nm, D in [('이용자',T), ('인력',P), ('기관(근사)',I), ('지역',R)]:
    print(f'  {nm:12s} {len(D):>8,}행  {list(D.columns)}')

---
# S4. 아티팩트 저장

In [ ]:
FEAT_ORDER = [
 # A군 — 구조 (라벨 불사용)
 'G_TRGTER_DEG','G_PRHNF_DEG','G_PAIR_CNT','G_PAIR_SHARE_TRGTER','G_PAIR_SHARE_PRHNF',
 'G_PRHNF_HHI','G_COMP_SIZE','G_PRHNF_TIME_OVERLAP','G_PRHNF_SAME_TIME_PAY',
 'G_INST_PRHNF_CNT','G_INST_TRGTER_CNT','G_REGION_PRHNF_CNT','G_REGION_TRGTER_CNT',
 # C군 — 중심성 · 커뮤니티
 'G_PRHNF_EIGCENT','G_TRGTER_EIGCENT','G_COMMUNITY_SIZE',
 # B군 — 라벨 사용 (LOO)
 'G_PRHNF_FRAUD_NBR','G_TRGTER_FRAUD_NBR','G_COMP_FRAUD_RATE',
 'G_COMMUNITY_FRAUD_RATE','G_INST_FRAUD_NBR','G_REGION_FRAUD_NBR',
 # 플래그
 'IS_NEW_TRGTER','IS_NEW_PRHNF','IS_NEW_INST',
]

ART = {
 'version':'v2', 'built_from':'train.parquet only',
 'manifest_checksum': MAN['checksums']['train'],
 'inst_key':'PRINST_TY|PRINST_LOCPLC_SIGNGU_CODE (근사 — 정확한 기관 식별 아님)',
 'same_pay_window_sec': 600,
 'trgter': T[['TRGTER_ID','t_deg','t_cnt','t_pos','comp_size','c_cnt','c_pos',
              'com_size','m_cnt','m_pos','t_eig']],
 'prhnf' : P[['PRHNF_ID','p_deg','p_cnt','p_pos','p_hhi','p_time_ov','p_same_pay',
              'comp_size','c_cnt','c_pos','com_size','m_cnt','m_pos','p_eig']],
 'pair'  : E[['TRGTER_ID','PRHNF_ID','pair_cnt','pair_pos']],
 'inst'  : I, 'region': R,
 'global_pos': GLOBAL_POS,
 'feature_order': FEAT_ORDER,
 'defaults': {
  'G_TRGTER_DEG':float(T.t_deg.median()), 'G_PRHNF_DEG':float(P.p_deg.median()),
  'G_PAIR_CNT':float(E.pair_cnt.median()), 'G_PAIR_SHARE_TRGTER':1.0,
  'G_PAIR_SHARE_PRHNF':float((E.pair_cnt/E.p_cnt).median()),
  'G_PRHNF_HHI':float(P.p_hhi.median()), 'G_COMP_SIZE':float(np.median(sz)),
  'G_PRHNF_TIME_OVERLAP':0.0, 'G_PRHNF_SAME_TIME_PAY':0.0,
  'G_INST_PRHNF_CNT':float(I.i_prhnf.median()), 'G_INST_TRGTER_CNT':float(I.i_trgter.median()),
  'G_REGION_PRHNF_CNT':float(R.r_prhnf.median()), 'G_REGION_TRGTER_CNT':float(R.r_trgter.median()),
  'G_PRHNF_EIGCENT':0.0, 'G_TRGTER_EIGCENT':0.0,
  'G_COMMUNITY_SIZE':float(csz.median()),
  'G_PRHNF_FRAUD_NBR':GLOBAL_POS, 'G_TRGTER_FRAUD_NBR':GLOBAL_POS,
  'G_COMP_FRAUD_RATE':GLOBAL_POS, 'G_COMMUNITY_FRAUD_RATE':GLOBAL_POS,
  'G_INST_FRAUD_NBR':GLOBAL_POS, 'G_REGION_FRAUD_NBR':GLOBAL_POS,
 },
}
with open(f'{OUT}/graph_feats.pkl','wb') as f: pickle.dump(ART, f)
print(f'저장 {OUT}/graph_feats.pkl ({os.path.getsize(f"{OUT}/graph_feats.pkl")/1e6:.1f} MB)')
print(f'피처 {len(FEAT_ORDER)}개')

---
# S5. `graph_predict.py` 생성 (가이드 5장-2)

In [ ]:
code = '''# -*- coding: utf-8 -*-
"""관계 네트워크 피처 적용 함수 — 운영 재현용 (v2)

artifact(graph_feats.pkl)는 train 결제만으로 구성한 노드 룩업 테이블이다.
축: 이용자 / 제공인력 / (이용자,인력) 조합 / 기관(근사) / 지역 / 연결요소 / 커뮤니티

기관 근사 키 = PRINST_TY|PRINST_LOCPLC_SIGNGU_CODE — 정확한 기관 식별이 아님.
신규 노드는 기본값 + IS_NEW_* 플래그로 처리한다.
운영 시 룩업 테이블은 최근 이력 배치로 주기 재계산한다.
"""
import pickle
import numpy as np
import pandas as pd


def add_graph_feats(df, artifact_path, leave_one_out=False):
    """df에 G_* 관계 피처를 붙여 반환.

    df            : TRGTER_ID, PRHNF_ID, PRINST_TY, PRINST_LOCPLC_SIGNGU_CODE 포함
    artifact_path : graph_feats.pkl 경로
    leave_one_out : True면 라벨 피처에서 자기 조합 몫 제외 (train 행 전용)
    """
    with open(artifact_path, 'rb') as f:
        art = pickle.load(f)
    T, P, E = art['trgter'], art['prhnf'], art['pair']
    I, R = art['inst'], art['region']
    dv, gp = art['defaults'], art['global_pos']

    o = df[['TRGTER_ID', 'PRHNF_ID']].copy()
    o['_INST'] = df['PRINST_TY'].astype(str) + '|' + df['PRINST_LOCPLC_SIGNGU_CODE'].astype(str)
    o['_REGION'] = df['PRINST_LOCPLC_SIGNGU_CODE'].astype(str)

    o['IS_NEW_TRGTER'] = (~o.TRGTER_ID.isin(set(T.TRGTER_ID))).astype(np.int8)
    o['IS_NEW_PRHNF'] = (~o.PRHNF_ID.isin(set(P.PRHNF_ID))).astype(np.int8)
    o['IS_NEW_INST'] = (~o._INST.isin(set(I._INST))).astype(np.int8)

    o = o.merge(T, on='TRGTER_ID', how='left')
    o = o.merge(P, on='PRHNF_ID', how='left', suffixes=('_t', '_p'))
    o = o.merge(E, on=['TRGTER_ID', 'PRHNF_ID'], how='left')
    o = o.merge(I, on='_INST', how='left')
    o = o.merge(R, on='_REGION', how='left')

    pc = o['pair_cnt'].fillna(0.0)
    pp = o['pair_pos'].fillna(0.0)
    tc, tp_ = o['t_cnt'], o['t_pos']
    pn, po = o['p_cnt'], o['p_pos']
    cc, cp_ = o['c_cnt_p'].fillna(o['c_cnt_t']), o['c_pos_p'].fillna(o['c_pos_t'])
    mc, mp_ = o['m_cnt_p'].fillna(o['m_cnt_t']), o['m_pos_p'].fillna(o['m_pos_t'])
    ic, ip_ = o['i_cnt'], o['i_pos']
    rc, rp_ = o['r_cnt'], o['r_pos']

    if leave_one_out:
        tc, tp_ = tc - pc, tp_ - pp
        pn, po = pn - pc, po - pp
        cc, cp_ = cc - pc, cp_ - pp
        mc, mp_ = mc - pc, mp_ - pp
        ic, ip_ = ic - pc, ip_ - pp
        rc, rp_ = rc - pc, rp_ - pp

    def safe(num, den):
        return np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)

    g = pd.DataFrame(index=o.index)
    g['G_TRGTER_DEG'] = o['t_deg']
    g['G_PRHNF_DEG'] = o['p_deg']
    g['G_PAIR_CNT'] = o['pair_cnt']
    g['G_PAIR_SHARE_TRGTER'] = safe(o['pair_cnt'], o['t_cnt'])
    g['G_PAIR_SHARE_PRHNF'] = safe(o['pair_cnt'], o['p_cnt'])
    g['G_PRHNF_HHI'] = o['p_hhi']
    g['G_COMP_SIZE'] = o['comp_size_p'].fillna(o['comp_size_t'])
    g['G_PRHNF_TIME_OVERLAP'] = o['p_time_ov']
    g['G_PRHNF_SAME_TIME_PAY'] = o['p_same_pay']
    g['G_INST_PRHNF_CNT'] = o['i_prhnf']
    g['G_INST_TRGTER_CNT'] = o['i_trgter']
    g['G_REGION_PRHNF_CNT'] = o['r_prhnf']
    g['G_REGION_TRGTER_CNT'] = o['r_trgter']
    g['G_PRHNF_EIGCENT'] = o['p_eig']
    g['G_TRGTER_EIGCENT'] = o['t_eig']
    g['G_COMMUNITY_SIZE'] = o['com_size_p'].fillna(o['com_size_t'])
    g['G_PRHNF_FRAUD_NBR'] = safe(po, pn)
    g['G_TRGTER_FRAUD_NBR'] = safe(tp_, tc)
    g['G_COMP_FRAUD_RATE'] = safe(cp_, cc)
    g['G_COMMUNITY_FRAUD_RATE'] = safe(mp_, mc)
    g['G_INST_FRAUD_NBR'] = safe(ip_, ic)
    g['G_REGION_FRAUD_NBR'] = safe(rp_, rc)
    g['IS_NEW_TRGTER'] = o['IS_NEW_TRGTER']
    g['IS_NEW_PRHNF'] = o['IS_NEW_PRHNF']
    g['IS_NEW_INST'] = o['IS_NEW_INST']

    for c in art['feature_order']:
        if c.startswith('IS_NEW'):
            continue
        g[c] = g[c].fillna(dv[c]).astype(np.float32)

    g = g[art['feature_order']]
    head = df[['row_id']].reset_index(drop=True) if 'row_id' in df.columns \
        else pd.DataFrame(index=range(len(df)))
    return pd.concat([head, g.reset_index(drop=True)], axis=1)
'''
with open(f'{OUT}/graph_predict.py','w',encoding='utf-8') as f: f.write(code)
import sys, importlib
sys.path.insert(0, OUT)
import graph_predict; importlib.reload(graph_predict)
from graph_predict import add_graph_feats
print('생성 · 임포트 완료')

---
# S6. 3분할 적용 · 저장

In [ ]:
AP = f'{OUT}/graph_feats.pkl'
F = {}
for nm, d, loo in [('train',tr,True), ('valid',va,False), ('test',te,False)]:
    g = add_graph_feats(d, AP, leave_one_out=loo)
    assert len(g)==len(d) and g.isna().sum().sum()==0
    assert (g.row_id.values == d.row_id.values).all()
    g.to_parquet(f'{OUT}/graph_feats_{nm}.parquet', index=False)
    F[nm] = g
    print(f'{nm:6s} {g.shape}  신규 이용자 {g.IS_NEW_TRGTER.mean():5.1%} · '
          f'인력 {g.IS_NEW_PRHNF.mean():5.1%} · 기관 {g.IS_NEW_INST.mean():5.1%}')

chk = add_graph_feats(va, AP, leave_one_out=False)
print('\n재현 검증 (가이드 5장-2):', chk.equals(pd.read_parquet(f'{OUT}/graph_feats_valid.parquet')))

---
# S7. 기여도 점검 — 축별 · 누수 통제

In [ ]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_fscore_support

DROP = {'row_id','_y','_INST','_REGION','INJSTUSE_AT','TRGTER_ID','PRHNF_ID','DELNG_DT',
        'DELNG_DE','PREDELNG_DE','SVC_BEGIN_DT','SVC_END_DT','CONFM_NO','DELNG_SN','PRINST_ID'}
num = [c for c in tr.columns if c not in DROP and pd.api.types.is_numeric_dtype(tr[c])]

trp2 = set(map(tuple, tr[['TRGTER_ID','PRHNF_ID']].values))
pair_seen = pd.Series(list(map(tuple, va[['TRGTER_ID','PRHNF_ID']].values))).isin(trp2).values

def run(cols, mask=None, verbose=False):
    Xtr = pd.concat([tr[num].reset_index(drop=True), F['train'][cols]], axis=1) if cols else tr[num]
    Xva = pd.concat([va[num].reset_index(drop=True), F['valid'][cols]], axis=1) if cols else va[num]
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, verbose=-1, random_state=42)
    m.fit(Xtr, tr._y, eval_set=[(Xva, va._y)], callbacks=[lgb.early_stopping(30, verbose=False)])
    p = m.predict_proba(Xva)[:,1]
    mk = np.ones(len(va), bool) if mask is None else mask
    yy, pp = va._y.values[mk], p[mk]
    ap = average_precision_score(yy, pp)
    f1 = max(precision_recall_fscore_support(yy,(pp>=t).astype(int),average='binary',zero_division=0)[2]
             for t in np.arange(0.05,0.96,0.01))
    return ap, f1, m

GA_STRUCT = FEAT_ORDER[:13]
GC_CENT   = FEAT_ORDER[13:16]
GB_LABEL  = FEAT_ORDER[16:22]
FLAGS     = FEAT_ORDER[22:]
SETS = {
 '기존만'                : [],
 'A군 구조 전체'          : GA_STRUCT + FLAGS,
 'A군 − G_PAIR_*'        : [c for c in GA_STRUCT if not c.startswith('G_PAIR')] + FLAGS,
 'C군 중심성·커뮤니티'      : GC_CENT + FLAGS,
 'B군 라벨 전체'          : GB_LABEL + FLAGS,
 'B군 − 기관·지역'        : ['G_PRHNF_FRAUD_NBR','G_TRGTER_FRAUD_NBR','G_COMP_FRAUD_RATE',
                          'G_COMMUNITY_FRAUD_RATE'] + FLAGS,
 '기관·지역 축만'          : ['G_INST_PRHNF_CNT','G_INST_TRGTER_CNT','G_REGION_PRHNF_CNT',
                          'G_REGION_TRGTER_CNT','G_INST_FRAUD_NBR','G_REGION_FRAUD_NBR'] + FLAGS,
 '전체'                  : FEAT_ORDER,
 '전체 − G_PAIR_*'        : [c for c in FEAT_ORDER if not c.startswith('G_PAIR')],
}

b_all = run([])[0]; b_np = run([], ~pair_seen)[0]
print(f'valid 전체 기준선 {va._y.mean():.4f} · 조합 미존재 기준선 {va._y.values[~pair_seen].mean():.4f}\n')
print(f'{"구성":24s}{"전체 valid":>20}{"조합 미존재":>20}')
print('-'*66)
RES = {}
for nm, cols in SETS.items():
    a = run(cols)[0]; b = run(cols, ~pair_seen)[0]
    RES[nm] = (a, b)
    print(f'{nm:24s}{a:>10.4f} ({a-b_all:+.4f}){b:>10.4f} ({b-b_np:+.4f})')

In [ ]:
#@title S7-2. 개별 피처 단독 판별력 · 중요도
_, _, mf = run(FEAT_ORDER)
imp = pd.DataFrame({'피처': mf.feature_name_, '중요도': mf.feature_importances_})
imp['순위'] = imp.중요도.rank(ascending=False).astype(int)
print('=== G_* 중요도 ===')
print(imp[imp.피처.str.startswith(('G_','IS_NEW'))].sort_values('중요도',ascending=False).to_string(index=False))

print('\n=== 단독 판별력 (valid 전체 / 조합 미존재) ===')
base_np = va._y.values[~pair_seen].mean()
for c in FEAT_ORDER:
    v = F['valid'][c].values
    if np.std(v) < 1e-9:
        print(f'  {c:24s} 상수'); continue
    a1 = max(average_precision_score(va._y, v), average_precision_score(va._y, -v))
    yv2, v2 = va._y.values[~pair_seen], v[~pair_seen]
    a2 = max(average_precision_score(yv2, v2), average_precision_score(yv2, -v2))
    print(f'  {c:24s} {a1:.4f} ({a1/va._y.mean():5.2f}배)   {a2:.4f} ({a2/base_np:5.2f}배)')

In [ ]:
#@title S7-3. 조기 종료 지표를 PR-AUC로 교체 후 재실행 (S7 결함 보정)
RES_LOGLOSS = dict(RES)   # 기존(binary_logloss 기준) 결과 보존

def run(cols, mask=None, verbose=False):
    """조기 종료를 average_precision 기준으로. n_estimators/patience도 상향."""
    Xtr = pd.concat([tr[num].reset_index(drop=True), F['train'][cols]], axis=1) if cols else tr[num]
    Xva = pd.concat([va[num].reset_index(drop=True), F['valid'][cols]], axis=1) if cols else va[num]
    m = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, verbose=-1, random_state=42,
                           metric='average_precision')
    m.fit(Xtr, tr._y, eval_set=[(Xva, va._y)],
          eval_metric='average_precision',
          callbacks=[lgb.early_stopping(50, verbose=False)])
    p = m.predict_proba(Xva)[:, 1]
    mk = np.ones(len(va), bool) if mask is None else mask
    yy, pp = va._y.values[mk], p[mk]
    ap = average_precision_score(yy, pp)
    f1 = max(precision_recall_fscore_support(yy, (pp >= t).astype(int),
             average='binary', zero_division=0)[2] for t in np.arange(0.05, 0.96, 0.01))
    return ap, f1, m, m.best_iteration_

b_all = run([])[0]; b_np = run([], ~pair_seen)[0]
print(f'valid 전체 기준선 {va._y.mean():.4f} · 조합 미존재 기준선 {va._y.values[~pair_seen].mean():.4f}')
print(f'조기 종료 지표: average_precision (기존: binary_logloss)\n')

print(f'{"구성":24s}{"전체 valid":>18}{"조합 미존재":>18}{"트리수":>8}{"기존대비":>10}')
print('-' * 80)
RES = {}
for nm, cols in SETS.items():
    a, _, _, ni = run(cols)
    b = run(cols, ~pair_seen)[0]
    RES[nm] = (a, b)
    d = a - RES_LOGLOSS[nm][0]
    print(f'{nm:24s}{a:>9.4f} ({a-b_all:+.4f}){b:>9.4f} ({b-b_np:+.4f}){ni:>8}{d:>+10.4f}')

print('\n=== 권고안 판정 ===')
best_np = max(RES.items(), key=lambda kv: kv[1][1])
print(f'조합 미존재 구간 최고: {best_np[0]}  {best_np[1][1]:.4f} ({best_np[1][1]-b_np:+.4f})')

In [ ]:
#@title S7-4. 공정 비교 — 조기 종료 제거 · 고정 트리 · seed 변동폭 측정
import time
N_TREES, SEEDS = 600, [42, 7, 2024]

def fit_eval(cols, seed):
    Xtr = pd.concat([tr[num].reset_index(drop=True), F['train'][cols]], axis=1) if cols else tr[num]
    Xva = pd.concat([va[num].reset_index(drop=True), F['valid'][cols]], axis=1) if cols else va[num]
    m = lgb.LGBMClassifier(n_estimators=N_TREES, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, verbose=-1, random_state=seed)
    m.fit(Xtr, tr._y)                      # 조기 종료 없음 — 전 구성 동일 학습량
    p = m.predict_proba(Xva)[:, 1]
    return (average_precision_score(va._y, p),
            average_precision_score(va._y.values[~pair_seen], p[~pair_seen]))

t0 = time.time()
STAT = {}
for nm, cols in SETS.items():
    r = np.array([fit_eval(cols, s) for s in SEEDS])
    STAT[nm] = (r[:,0].mean(), r[:,0].std(), r[:,1].mean(), r[:,1].std())
    print(f'{nm:24s} 완료 ({time.time()-t0:.0f}s)')

ba, _, bn, _ = STAT['기존만']
print(f'\n트리 {N_TREES} 고정 · seed {len(SEEDS)}개 평균±표준편차')
print(f'기준선 — 전체 {va._y.mean():.4f} · 조합 미존재 {va._y.values[~pair_seen].mean():.4f}\n')
print(f'{"구성":24s}{"전체 valid":>26}{"조합 미존재":>26}')
print('-' * 78)
for nm, (a, sa, b, sb) in STAT.items():
    print(f'{nm:24s}{a:>8.4f}±{sa:.4f} ({a-ba:+.4f}){b:>8.4f}±{sb:.4f} ({b-bn:+.4f})')

print('\n=== 판정 ===')
noise = max(v[3] for v in STAT.values()) * 2      # seed 표준편차 2배 = 노이즈 하한
print(f'조합 미존재 구간 seed 노이즈(2σ 최대): ±{noise:.4f}')
for nm, (_, _, b, _) in sorted(STAT.items(), key=lambda kv: -kv[1][2]):
    d = b - bn
    v = '유의' if d > noise else ('노이즈 수준' if d > -noise else '악화')
    print(f'  {nm:24s} {d:+.4f}  → {v}')

In [ ]:
#@title S7-5. 최종 확정 — 유의 구성 결합 · 중복 피처 제거
B_CORE  = ['G_PRHNF_FRAUD_NBR','G_TRGTER_FRAUD_NBR','G_COMP_FRAUD_RATE']
A_CORE  = [c for c in FEAT_ORDER[:13]
           if not c.startswith('G_PAIR') and 'INST' not in c and 'REGION' not in c]
FLAGS2  = ['IS_NEW_TRGTER','IS_NEW_PRHNF']        # IS_NEW_INST는 상수라 제외

FINAL = {
 '기존만'                    : [],
 'B군 − 기관·지역 (현 권고)'    : ['G_PRHNF_FRAUD_NBR','G_TRGTER_FRAUD_NBR','G_COMP_FRAUD_RATE',
                              'G_COMMUNITY_FRAUD_RATE'] + FEAT_ORDER[22:],
 'B핵심 (커뮤니티 중복 제거)'     : B_CORE + FLAGS2,
 'A핵심 + B핵심'             : A_CORE + B_CORE + FLAGS2,
 'A핵심만'                   : A_CORE + FLAGS2,
}
print('A핵심 =', A_CORE, '\n')

ST2 = {}
for nm, cols in FINAL.items():
    r = np.array([fit_eval(cols, s) for s in SEEDS])
    ST2[nm] = (r[:,0].mean(), r[:,0].std(), r[:,1].mean(), r[:,1].std(), len(cols))
    print(f'{nm:26s} 완료')

ba2, bn2 = ST2['     ㅜㅡ,'][0], ST2['기존만'][2]
print(f'\n{"구성":26s}{"피처":>5}{"전체 valid":>25}{"조합 미존재":>25}')
print('-' * 82)
for nm, (a, sa, b, sb, k) in ST2.items():
    print(f'{nm:26s}{k:>5}{a:>8.4f}±{sa:.4f} ({a-ba2:+.4f}){b:>8.4f}±{sb:.4f} ({b-bn2:+.4f})')

best = max(((k,v) for k,v in ST2.items() if k!='기존만'), key=lambda kv: kv[1][2])
print(f'\n최종 권고: {best[0]}  조합 미존재 {best[1][2]:.4f} ({best[1][2]-bn2:+.4f}, 피처 {best[1][4]}개)')

In [ ]:
#@title S7-5b. 출력만 (재학습 없음)
keys = list(ST2.keys())
ba2, bn2 = ST2[keys[0]][0], ST2[keys[0]][2]      # 첫 항목 = 기준선

print(f'{"구성":26s}{"피처":>5}{"전체 valid":>25}{"조합 미존재":>25}')
print('-' * 82)
for nm in keys:
    a, sa, b, sb, k = ST2[nm]
    print(f'{nm:26s}{k:>5}{a:>8.4f}±{sa:.4f} ({a-ba2:+.4f}){b:>8.4f}±{sb:.4f} ({b-bn2:+.4f})')

best = max(((k_, v) for k_, v in ST2.items() if k_ != keys[0]), key=lambda kv: kv[1][2])
print(f'\n최종 권고: {best[0]}  조합 미존재 {best[1][2]:.4f} '
      f'({best[1][2]-bn2:+.4f}, 피처 {best[1][4]}개)')

In [ ]:
#@title S9. 권고안 확정 — 아티팩트·정의서 갱신
RECOMMENDED = ['G_PRHNF_FRAUD_NBR','G_TRGTER_FRAUD_NBR','G_COMP_FRAUD_RATE',
               'IS_NEW_TRGTER','IS_NEW_PRHNF']

with open(f'{OUT}/graph_feats.pkl','rb') as f: ART = pickle.load(f)
ART['recommended'] = RECOMMENDED
ART['evidence'] = {
  'protocol': '600트리 고정 · 조기 종료 없음 · seed [42,7,2024] 평균',
  'eval_segment': 'valid 중 train에 조합이 없는 행 (n=30,976, 양성률 0.0175)',
  'baseline_prauc': 0.0921, 'recommended_prauc': 0.1250,
  'delta': 0.0329, 'seed_sd': 0.0037,
  'rejected': {'기관·지역 축': -0.0064, '전체 25개': -0.0348,
               'A핵심+B핵심': +0.0090, 'C군': +0.0061},
}
with open(f'{OUT}/graph_feats.pkl','wb') as f: pickle.dump(ART, f)

dfd = pd.read_excel(f'{OUT}/피처정의서_G_v2.xlsx')
dfd['권고'] = np.where(dfd.피처.isin(RECOMMENDED), '★ 채택', '')
dfd.loc[dfd.피처=='G_COMMUNITY_FRAUD_RATE','비고'] = 'G_COMP_FRAUD_RATE와 동일 — Louvain 미분할'
dfd.loc[dfd.피처=='IS_NEW_INST','비고'] = '상수(기관 100% 중복) — 제외'
dfd.loc[dfd.피처.str.contains('INST|REGION'),'비고'] += ' / 조합 미존재 구간 −0.0064, 제외'
dfd.to_excel(f'{OUT}/피처정의서_G_v2.xlsx', index=False)
print(dfd[['피처','군','권고','비고']].to_string(index=False))

for nm in ['train','valid','test']:
    F[nm][RECOMMENDED].assign(row_id=[tr,va,te][['train','valid','test'].index(nm)].row_id.values) \
     [['row_id']+RECOMMENDED].to_parquet(f'{OUT}/graph_feats_{nm}_recommended.parquet', index=False)
print(f'\n권고안 전용 피처 테이블 3개 저장 완료 (5개 컬럼)')

In [ ]:
#@title S9-fix. 비고 열 정정
dfd = pd.read_excel(f'{OUT}/피처정의서_G_v2.xlsx')
dfd['비고'] = ''          # NaN 초기화 후 전면 재작성

NOTE = {
 'G_PAIR_CNT'            : '조합 중복 구간에서 라벨 대리 — 단독 9.85배 / 조합 미존재 1.00배. 사용 금지',
 'G_PAIR_SHARE_TRGTER'   : '조합 축 파생 — 조합 미존재 구간 신호 없음',
 'G_PAIR_SHARE_PRHNF'    : '조합 축 파생 — 조합 미존재 구간 신호 없음',
 'G_PRHNF_TIME_OVERLAP'  : '발생 인력 45명(0.1%) — 희소, 미채택',
 'G_PRHNF_SAME_TIME_PAY' : 'DELNG_DT 10분 창. 발생 인력 167명(0.2%) — 희소, 미채택',
 'G_PRHNF_EIGCENT'       : 'C군 — 조합 미존재 +0.0061(노이즈 수준), 미채택',
 'G_TRGTER_EIGCENT'      : 'C군 — 조합 미존재 +0.0061(노이즈 수준), 미채택',
 'G_COMMUNITY_SIZE'      : 'Louvain 미분할(연결요소=커뮤니티) — 중요도 0',
 'G_COMMUNITY_FRAUD_RATE': 'G_COMP_FRAUD_RATE와 값 동일(Louvain 미분할) — 중요도 0',
 'IS_NEW_INST'           : '기관 중복률 100%로 상수 — 제외',
}
INST_AGG = ['G_INST_PRHNF_CNT','G_INST_TRGTER_CNT',
            'G_REGION_PRHNF_CNT','G_REGION_TRGTER_CNT',
            'G_INST_FRAUD_NBR','G_REGION_FRAUD_NBR']

dfd['비고'] = dfd.피처.map(NOTE).fillna('')
dfd.loc[dfd.피처.isin(INST_AGG), '비고'] = (
    '기관 근사 키(유형×시군구) — 정확한 기관 식별 아님. '
    '기관·지역 축 단독 조합 미존재 −0.0064 → 제외')
m = (dfd.군 == 'B') & (dfd.권고 == '★ 채택')
dfd.loc[m, '비고'] = 'selective labels 한계 — 조사 이력 많은 주체를 위험하게 학습. 운영 시 재점검 필요'

dfd.to_excel(f'{OUT}/피처정의서_G_v2.xlsx', index=False)
print(dfd[['피처','군','권고','비고']].to_string(index=False))

In [ ]:
#@title S9-fix2. 권고 열 복원 + NaN 재발 차단
dfd = pd.read_excel(f'{OUT}/피처정의서_G_v2.xlsx', keep_default_na=False)
dfd['권고'] = np.where(dfd.피처.isin(RECOMMENDED), '★ 채택', '—')
dfd = dfd.fillna('')
dfd.to_excel(f'{OUT}/피처정의서_G_v2.xlsx', index=False)

chk = pd.read_excel(f'{OUT}/피처정의서_G_v2.xlsx', keep_default_na=False)
print(chk[['피처','군','권고','비고']].to_string(index=False))
print(f"\n채택 {(chk.권고=='★ 채택').sum()}개 · NaN {chk.isna().sum().sum()}개")

---
# S8. 단독 점수 · 피처 정의서

In [ ]:
mg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                        n_jobs=-1, verbose=-1, random_state=42)
mg.fit(F['train'][FEAT_ORDER], tr._y, eval_set=[(F['valid'][FEAT_ORDER], va._y)],
       callbacks=[lgb.early_stopping(30, verbose=False)])
for nm, d in [('train',tr), ('valid',va), ('test',te)]:
    s = mg.predict_proba(F[nm][FEAT_ORDER])[:,1]
    pd.DataFrame({'row_id': d.row_id.values, 'y': d._y, 'score_graph': s}) \
      .to_csv(f'{OUT}/graph_scores_{nm}.csv', index=False)
sv = mg.predict_proba(F['valid'][FEAT_ORDER])[:,1]
print(f"score_graph valid PR-AUC {average_precision_score(va._y, sv):.4f} "
      f"· 조합 미존재 {average_precision_score(va._y.values[~pair_seen], sv[~pair_seen]):.4f}")

In [ ]:
DEF = [
 ('G_TRGTER_DEG','A','이용자의 distinct 제공인력 수','엣지에서 TRGTER_ID별 PRHNF_ID nunique','인력 갈아타기·명의 대여'),
 ('G_PRHNF_DEG','A','제공인력의 distinct 이용자 수','엣지에서 PRHNF_ID별 TRGTER_ID nunique','과다 수임'),
 ('G_PAIR_CNT','A','이용자–인력 조합의 결제 건수','(TRGTER_ID,PRHNF_ID) 그룹 크기','특정 조합 반복(담합 기본 신호)'),
 ('G_PAIR_SHARE_TRGTER','A','조합 건수 ÷ 이용자 전체 건수','pair_cnt / t_cnt','이용자 관점 조합 집중도'),
 ('G_PAIR_SHARE_PRHNF','A','조합 건수 ÷ 인력 전체 건수','pair_cnt / p_cnt','인력 관점 조합 집중도'),
 ('G_PRHNF_HHI','A','인력의 이용자별 결제 비중 제곱합','Σ(pair_cnt/p_cnt)²','소수 이용자 몰아주기'),
 ('G_COMP_SIZE','A','노드가 속한 연결요소 크기','connected_components','폐쇄적 소집단'),
 ('G_PRHNF_TIME_OVERLAP','A','같은 인력의 서비스 시간대 겹침 건수(타 이용자)','SVC_BEGIN_DT~SVC_END_DT 구간 스윕','물리적으로 불가능한 동시 서비스'),
 ('G_PRHNF_SAME_TIME_PAY','A','같은 인력의 10분 내 타 이용자 결제 건수','DELNG_DT 슬라이딩 윈도우 600초','일괄·대리 결제 의심'),
 ('G_INST_PRHNF_CNT','A','기관(근사)의 제공인력 수','_INST별 PRHNF_ID nunique','기관 규모 대비 이상'),
 ('G_INST_TRGTER_CNT','A','기관(근사)의 이용자 수','_INST별 TRGTER_ID nunique','기관 규모'),
 ('G_REGION_PRHNF_CNT','A','지역의 제공인력 수','시군구별 PRHNF_ID nunique','지역 규모'),
 ('G_REGION_TRGTER_CNT','A','지역의 이용자 수','시군구별 TRGTER_ID nunique','지역 규모'),
 ('G_PRHNF_EIGCENT','C','인력 사영 그래프 고유벡터 중심성','BᵀB 최대 고유벡터, 최댓값 정규화','네트워크 중심 인력'),
 ('G_TRGTER_EIGCENT','C','이용자 사영 그래프 고유벡터 중심성','BBᵀ 최대 고유벡터','네트워크 중심 이용자'),
 ('G_COMMUNITY_SIZE','C','Louvain 커뮤니티 크기','크기 10 이상 연결요소에 Louvain, 그 외 연결요소','군집 규모'),
 ('G_PRHNF_FRAUD_NBR','B','인력과 연결된 (자신 제외) 이용자들의 부정률','(p_pos−pair_pos)/(p_cnt−pair_cnt)','인력 단위 위험도'),
 ('G_TRGTER_FRAUD_NBR','B','이용자와 연결된 (자신 제외) 인력들의 부정 연루율','(t_pos−pair_pos)/(t_cnt−pair_cnt)','이용자 단위 위험도'),
 ('G_COMP_FRAUD_RATE','B','연결요소 내 (자기 조합 제외) 부정률','(c_pos−pair_pos)/(c_cnt−pair_cnt)','소집단 위험도'),
 ('G_COMMUNITY_FRAUD_RATE','B','커뮤니티 내 (자기 조합 제외) 부정률','(m_pos−pair_pos)/(m_cnt−pair_cnt)','커뮤니티 위험도'),
 ('G_INST_FRAUD_NBR','B','기관(근사) 내 (자기 조합 제외) 부정률','(i_pos−pair_pos)/(i_cnt−pair_cnt)','기관 단위 위험도'),
 ('G_REGION_FRAUD_NBR','B','지역 내 (자기 조합 제외) 부정률','(r_pos−pair_pos)/(r_cnt−pair_cnt)','지역 단위 위험도'),
 ('IS_NEW_TRGTER','—','train 그래프에 없는 신규 이용자','룩업 실패 시 1','cold-start'),
 ('IS_NEW_PRHNF','—','train 그래프에 없는 신규 제공인력','룩업 실패 시 1','cold-start'),
 ('IS_NEW_INST','—','train에 없는 신규 기관(근사)','룩업 실패 시 1','cold-start'),
]
dfd = pd.DataFrame(DEF, columns=['피처','군','정의','산출식','탐지 의도'])
dfd['기본값'] = dfd.피처.map(lambda c: round(ART['defaults'].get(c,0),4))
dfd['비고'] = ''
dfd.loc[dfd.피처.str.contains('INST|REGION'),'비고'] = '기관 근사 키(유형×시군구) — 정확한 기관 식별 아님'
dfd.loc[dfd.군=='B','비고'] = 'selective labels 한계 — 조사 빈도 반영 가능'
dfd.to_excel(f'{OUT}/피처정의서_G_v2.xlsx', index=False)
print(dfd[['피처','군','정의','기본값']].to_string(index=False))
print('\n산출물:')
for f in sorted(os.listdir(OUT)):
    if f != '__pycache__':
        print(f'  {os.path.getsize(f"{OUT}/{f}")/1e6:8.2f} MB  {f}')

---
## 가이드 대조

| 항목 | 이행 |
|---|---|
| 2장 `DELNG_DT` 동시결제 | `G_PRHNF_SAME_TIME_PAY` |
| 2장 기관 근사 키 | `_INST` = `PRINST_TY`&#124;`PRINST_LOCPLC_SIGNGU_CODE`, 문서에 근사임을 명시 |
| 3장 1~5 누수 방지 | train만 그래프 구성 · 룩업만 · 기본값+플래그 · train 라벨만+LOO · 분할 유지 |
| 3장 인력·기관·지역 축 | 4개 축 전부 구현, S0-2에서 축별 중복률 실측 |
| 4장 A군 8개 | 전부 |
| 4장 B군 3개 | 전부 + 기관·지역·커뮤니티 3개 추가 |
| 4장 C군 | 고유벡터 중심성 2개 + Louvain 커뮤니티 2개 |
| 5장 산출물 4종 | 피처 테이블 3개 · pkl · py · scores · 정의서 |
| 6장 test | 미사용 (최종 확정 후 1회) |

In [ ]:
#@title 실험 결과 저장 (세션 종료 전 1회)
import pickle
with open(f'{OUT}/experiments.pkl','wb') as f:
    pickle.dump({'STAT': STAT, 'ST2': ST2, 'SETS': SETS, 'FINAL': FINAL,
                 'RES_LOGLOSS': RES_LOGLOSS, 'N_TREES': N_TREES, 'SEEDS': SEEDS}, f)
print('저장 완료 — 다음 세션에서 재학습 불필요')

In [ ]:
#@title T1. PL 판정 기준으로 사전 확인 (valid, test 미개봉)
new_t = (F['valid'].IS_NEW_TRGTER.values == 1)
SEG = {'전체 valid': np.ones(len(va), bool),
       '조합 미존재': ~pair_seen,
       '신규 이용자': new_t}
print({k: f'n={v.sum():,} 양성률 {va._y.values[v].mean():.4f}' for k, v in SEG.items()})

def fit_eval3(cols, seed):
    Xtr = pd.concat([tr[num].reset_index(drop=True), F['train'][cols]], axis=1) if cols else tr[num]
    Xva = pd.concat([va[num].reset_index(drop=True), F['valid'][cols]], axis=1) if cols else va[num]
    m = lgb.LGBMClassifier(n_estimators=N_TREES, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, verbose=-1, random_state=seed)
    m.fit(Xtr, tr._y)
    p = m.predict_proba(Xva)[:, 1]
    out = {}
    for sn, mk in SEG.items():
        yy, pp = va._y.values[mk], p[mk]
        ap = average_precision_score(yy, pp)
        f1s = [(precision_recall_fscore_support(yy, (pp >= t).astype(int),
                average='binary', zero_division=0)[2], t) for t in np.arange(0.02, 0.98, 0.01)]
        f1, thr = max(f1s)
        out[sn] = (ap, f1, thr)
    return out

CAND = {'기존만': [],
        'B핵심 5개 (권고)': RECOMMENDED,
        'A핵심 + B핵심': A_CORE + B_CORE + FLAGS2,
        'A핵심만': A_CORE + FLAGS2,
        'B군 + 커뮤니티 7개': ['G_PRHNF_FRAUD_NBR','G_TRGTER_FRAUD_NBR','G_COMP_FRAUD_RATE',
                            'G_COMMUNITY_FRAUD_RATE','IS_NEW_TRGTER','IS_NEW_PRHNF','IS_NEW_INST']}

T1 = {}
for nm, cols in CAND.items():
    rs = [fit_eval3(cols, s) for s in SEEDS]
    T1[nm] = {sn: (np.mean([r[sn][0] for r in rs]), np.std([r[sn][0] for r in rs]),
                   np.mean([r[sn][1] for r in rs]), np.std([r[sn][1] for r in rs]),
                   np.mean([r[sn][2] for r in rs])) for sn in SEG}
    print(f'{nm:22s} 완료')

for sn in SEG:
    b_ap, b_f1 = T1['기존만'][sn][0], T1['기존만'][sn][2]
    print(f'\n=== {sn} (기준선 PR-AUC {b_ap:.4f} · F1 {b_f1:.4f}) ===')
    print(f'{"구성":22s}{"PR-AUC":>22}{"F1":>22}{"임계":>7}')
    for nm in CAND:
        ap, sa, f1, sf, thr = T1[nm][sn]
        print(f'{nm:22s}{ap:>8.4f}±{sa:.4f}({ap-b_ap:+.4f}){f1:>8.4f}±{sf:.4f}({f1-b_f1:+.4f}){thr:>7.2f}')

print('\n=== PL 판정 기준(신규 이용자 · F1) 순위 ===')
for nm, v in sorted(T1.items(), key=lambda kv: -kv[1]['신규 이용자'][2]):
    d = v['신규 이용자'][2] - T1['기존만']['신규 이용자'][2]
    print(f'  {nm:22s} F1 {v["신규 이용자"][2]:.4f} ({d:+.4f})  seed σ {v["신규 이용자"][3]:.4f}')

In [ ]:
#@title 라벨 복구 (재개 셀 버그 수정)
print('dtype:', tr.INJSTUSE_AT.dtype, '| 값:', tr.INJSTUSE_AT.unique()[:5])

def to_y(s):
    return (s.astype(str).str.upper()=='Y').astype(np.int8).values if s.dtype==object \
           else s.values.astype(np.int8)
for d in (tr, va, te):
    d['_y'] = to_y(d['INJSTUSE_AT'])

print(f'부정률  train {tr._y.mean():.4f} / valid {va._y.mean():.4f} / test {te._y.mean():.4f}')
assert va._y.mean() > 0, '라벨 여전히 이상'
print('→ 0.5332 / 0.0830 / 0.0830 이면 정상')

In [ ]:
#@title T2. test 1회 개봉 — 권고 5피처 확정 구성 (구간 매핑 수정)
import datetime
GUARD = f'{OUT}/TEST_OPENED.txt'
assert not os.path.exists(GUARD), f'test 이미 개봉됨 → {open(GUARD).read()}'

pair_te = pd.Series(list(map(tuple, te[['TRGTER_ID','PRHNF_ID']].values))).isin(_trp).values
SEG_TE = {'전체 test': np.ones(len(te), bool),
          '조합 미존재': ~pair_te,
          '신규 이용자': (F['test'].IS_NEW_TRGTER.values == 1)}
SEG_MAP = {'전체 test': '전체 valid', '조합 미존재': '조합 미존재', '신규 이용자': '신규 이용자'}
NAME_MAP = {'기존만': '기존만', '권고 5피처': 'B핵심 5개 (권고)'}
print({k: f'n={v.sum():,} 양성률 {te._y.values[v].mean():.4f}' for k, v in SEG_TE.items()})

def pk(y, p, k=100):
    return y[np.argsort(-p)[:k]].mean()

ROWS = []
for nm, cols in [('기존만', []), ('권고 5피처', RECOMMENDED)]:
    preds = []
    for s in SEEDS:
        Xtr = pd.concat([tr[num].reset_index(drop=True), F['train'][cols]], axis=1) if cols else tr[num]
        Xte = pd.concat([te[num].reset_index(drop=True), F['test'][cols]], axis=1) if cols else te[num]
        m = lgb.LGBMClassifier(n_estimators=N_TREES, learning_rate=0.05, num_leaves=63,
                               n_jobs=-1, verbose=-1, random_state=s)
        m.fit(Xtr, tr._y)
        preds.append(m.predict_proba(Xte)[:, 1])
    for sn, mk in SEG_TE.items():
        thr = T1[NAME_MAP[nm]][SEG_MAP[sn]][4]          # valid에서 정한 임계 고정
        v = []
        for p in preds:
            yy, pp = te._y.values[mk], p[mk]
            pr, rc, f1, _ = precision_recall_fscore_support(
                yy, (pp >= thr).astype(int), average='binary', zero_division=0)
            f1_max = max(precision_recall_fscore_support(yy, (pp >= t).astype(int),
                         average='binary', zero_division=0)[2] for t in np.arange(0.02, 0.98, 0.01))
            v.append([average_precision_score(yy, pp), pr, rc, f1, f1_max, pk(yy, pp, 100)])
        v = np.array(v)
        ROWS.append(dict(구성=nm, 구간=sn, n=int(mk.sum()), 양성률=te._y.values[mk].mean(),
                         임계=thr, **{k: v[:, i].mean() for i, k in enumerate(
                             ['PR_AUC','정밀도','재현율','F1','F1_최적','P@100'])},
                         PR_AUC_sd=v[:, 0].std(), F1_sd=v[:, 3].std()))
    print(f'{nm} 완료')

R = pd.DataFrame(ROWS)
for sn in SEG_TE:
    d = R[R.구간 == sn].set_index('구성')
    b, g = d.loc['기존만'], d.loc['권고 5피처']
    print(f'\n=== {sn} (n {b.n:,} · 양성률 {b.양성률:.4f} · 임계 {b.임계:.2f}→{g.임계:.2f}) ===')
    for c in ['PR_AUC','정밀도','재현율','F1','F1_최적','P@100']:
        print(f'  {c:8s} {b[c]:.4f} → {g[c]:.4f}  ({g[c]-b[c]:+.4f})')

R.round(4).to_csv(f'{OUT}/test_결과_권고5피처.csv', index=False, encoding='utf-8-sig')
open(GUARD, 'w').write(f'{datetime.datetime.now():%Y-%m-%d %H:%M} · 권고 5피처 · seed {SEEDS}')
print('\n저장: test_결과_권고5피처.csv · 개봉 기록 남김')

In [ ]:
#@title T3. seed 변동폭 확인 (재학습 없음)
print(R[['구성','구간','n','양성률','PR_AUC','PR_AUC_sd','F1','F1_sd','P@100']]
      .round(4).to_string(index=False))

print('\n=== 증분 대 노이즈 ===')
for sn in SEG_TE:
    d = R[R.구간==sn].set_index('구성')
    b, g = d.loc['기존만'], d.loc['권고 5피처']
    for c, sd in [('PR_AUC','PR_AUC_sd'), ('F1','F1_sd')]:
        delta, noise = g[c]-b[c], 2*max(b[sd], g[sd])
        v = '유의' if abs(delta) > noise else '노이즈 수준'
        print(f'  {sn:10s} {c:7s} {delta:+.4f}  (2σ ±{noise:.4f})  → {v}')